# Eco-Acoustic Mesh — Gemma 4 E2B Audio Fine-Tuning

This notebook fine-tunes **Gemma 4 E2B** for environmental sound classification in anti-poaching applications.

**Target Classes:** Chainsaw, Gunshot, Vehicle, Ambient

**Pipeline:**
1. Download ESC-50 environmental sound dataset
2. Filter and prepare relevant audio classes
3. Generate training pairs (audio + classification prompt)
4. Fine-tune Gemma 4 E2B using LoRA (via Unsloth)
5. Quantize to INT4 for edge deployment
6. Export GGUF for Ollama

> **Hardware:** T4 GPU (free Colab) or A100 (Colab Pro)

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q unsloth transformers datasets soundfile librosa
!pip install -q bitsandbytes accelerate peft trl
!pip install -q ollama llama-cpp-python

In [ ]:
import os
import json
import random
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from IPython.display import Audio, display

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 2. Download & Prepare ESC-50 Dataset

ESC-50 contains 2,000 environmental audio recordings across 50 classes.
We map relevant ESC-50 classes to our 4 target categories.

In [ ]:
# Download ESC-50
!git clone https://github.com/karolpiczak/ESC-50.git --depth 1
ESC50_DIR = Path("ESC-50")
AUDIO_DIR = ESC50_DIR / "audio"
META_FILE = ESC50_DIR / "meta" / "esc50.csv"

In [ ]:
import pandas as pd

meta = pd.read_csv(META_FILE)
print(f"Total samples: {len(meta)}")
print(f"\nAll classes:\n{meta['category'].value_counts().to_string()}")

# Map ESC-50 classes to our target categories
CLASS_MAPPING = {
    # CHAINSAW
    'chainsaw': 'CHAINSAW',
    'hand_saw': 'CHAINSAW',
    
    # GUNSHOT
    'gunshot': 'GUNSHOT',
    'fireworks': 'GUNSHOT',  # Similar impulsive sounds
    
    # VEHICLE
    'engine': 'VEHICLE',
    'train': 'VEHICLE',
    'helicopter': 'VEHICLE',
    'airplane': 'VEHICLE',
    
    # AMBIENT (natural sounds)
    'rain': 'AMBIENT',
    'sea_waves': 'AMBIENT',
    'crackling_fire': 'AMBIENT',
    'crickets': 'AMBIENT',
    'chirping_birds': 'AMBIENT',
    'wind': 'AMBIENT',
    'thunderstorm': 'AMBIENT',
    'water_drops': 'AMBIENT',
    'insects': 'AMBIENT',
    'frog': 'AMBIENT',
    'crow': 'AMBIENT',
    'hen': 'AMBIENT',
    'dog': 'AMBIENT',
    'rooster': 'AMBIENT',
    'cat': 'AMBIENT',
}

# Filter to mapped classes only
meta['target_class'] = meta['category'].map(CLASS_MAPPING)
dataset = meta.dropna(subset=['target_class']).copy()
print(f"\nFiltered dataset: {len(dataset)} samples")
print(dataset['target_class'].value_counts())

## 3. Audio Preprocessing

Gemma E2B requires: **16kHz, mono, float32 waveforms**

In [ ]:
TARGET_SR = 16000
MAX_DURATION = 5.0  # seconds — matches our sentinel buffer

def load_and_preprocess(filename):
    """Load audio, resample to 16kHz mono, normalize."""
    filepath = AUDIO_DIR / filename
    audio, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    
    # Pad or trim to exact duration
    target_length = int(TARGET_SR * MAX_DURATION)
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]
    
    # Normalize
    peak = np.max(np.abs(audio))
    if peak > 1e-6:
        audio = audio / peak
    
    return audio.astype(np.float32)

# Test with one sample
sample = dataset.iloc[0]
audio = load_and_preprocess(sample['filename'])
print(f"Sample: {sample['category']} -> {sample['target_class']}")
print(f"Audio shape: {audio.shape}, dtype: {audio.dtype}")
display(Audio(audio, rate=TARGET_SR))

In [ ]:
import io

def audio_to_wav_bytes(audio, sr=TARGET_SR):
    """Convert numpy audio to WAV bytes (in-memory)."""
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format='WAV', subtype='FLOAT')
    buf.seek(0)
    return buf.read()

# Prepare all audio samples
print("Processing all audio samples...")
processed = []
errors = 0

for idx, row in dataset.iterrows():
    try:
        audio = load_and_preprocess(row['filename'])
        wav_bytes = audio_to_wav_bytes(audio)
        processed.append({
            'filename': row['filename'],
            'audio': audio,
            'wav_bytes': wav_bytes,
            'target_class': row['target_class'],
            'original_class': row['category'],
        })
    except Exception as e:
        errors += 1

print(f"Processed: {len(processed)} samples, Errors: {errors}")

# Balance classes
from collections import Counter
class_counts = Counter(s['target_class'] for s in processed)
print(f"Class distribution: {dict(class_counts)}")

## 4. Create Training Dataset

Format: Each sample is a conversation with audio input + JSON classification output.

In [ ]:
SYSTEM_PROMPT = """You are an Environmental Sound Analyst deployed in a wildlife protection zone. Classify audio into: CHAINSAW, GUNSHOT, VEHICLE, or AMBIENT. Respond with JSON only."""

USER_PROMPT = """Classify this audio segment. Respond with JSON: {"class": "<CATEGORY>", "confidence": <0.0-1.0>}"""

def create_training_sample(sample):
    """Create a training conversation pair."""
    target = sample['target_class']
    
    # Generate realistic confidence based on audio quality
    if target == 'AMBIENT':
        confidence = round(random.uniform(0.85, 0.99), 2)
    else:
        confidence = round(random.uniform(0.75, 0.98), 2)
    
    # Reasoning descriptions for each class
    reasons = {
        'CHAINSAW': [
            'Sustained mechanical buzzing with harmonic overtones and periodic load variations',
            'High-frequency oscillating motor sound with characteristic two-stroke engine pattern',
            'Continuous cutting/sawing noise with varying pitch under load',
        ],
        'GUNSHOT': [
            'Sharp impulsive transient with rapid decay and distant echo',
            'High-amplitude ballistic crack followed by reverberant tail',
            'Sudden explosive onset with broadband energy burst',
        ],
        'VEHICLE': [
            'Low-frequency engine rumble with rhythmic combustion pattern',
            'Continuous motor noise with tire-on-surface friction harmonics',
            'Steady mechanical drone with Doppler shift suggesting movement',
        ],
        'AMBIENT': [
            'Natural environmental sounds with no mechanical or explosive components',
            'Organic acoustic patterns consistent with wildlife and weather',
            'Background noise with biophonic and geophonic signatures only',
        ],
    }
    
    response = json.dumps({
        'class': target,
        'confidence': confidence,
        'reasoning': random.choice(reasons[target]),
    })
    
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': USER_PROMPT},
            {'role': 'assistant', 'content': response},
        ],
        'audio_bytes': sample['wav_bytes'],
        'target_class': target,
    }

# Create full training dataset
train_data = [create_training_sample(s) for s in processed]
random.shuffle(train_data)

# Split train/val (90/10)
split_idx = int(len(train_data) * 0.9)
train_split = train_data[:split_idx]
val_split = train_data[split_idx:]

print(f"Training samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")
print(f"\nExample output:\n{train_data[0]['messages'][2]['content']}")

## 5. Load Gemma 4 E2B + LoRA Configuration

Using Unsloth for efficient 4-bit LoRA fine-tuning on free Colab GPUs.

In [ ]:
from unsloth import FastModel

# Load Gemma 4 E2B with 4-bit quantization
model, tokenizer = FastModel.from_pretrained(
    model_name="google/gemma-4-e2b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# Apply LoRA adapters
model = FastModel.get_peft_model(
    model,
    r=16,                   # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.2f}%)")

## 6. Training

Fine-tune with SFTTrainer using our audio classification dataset.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# Convert to HuggingFace dataset format (text only for now)
def format_for_training(sample):
    """Format as chat template string."""
    msgs = sample['messages']
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

train_texts = [format_for_training(s) for s in train_split]
val_texts = [format_for_training(s) for s in val_split]

train_ds = Dataset.from_list(train_texts)
val_ds = Dataset.from_list(val_texts)

print(f"Train dataset: {len(train_ds)} samples")
print(f"Val dataset: {len(val_ds)} samples")

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir="./eco_mesh_checkpoints",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")

## 7. Evaluation

In [ ]:
# Test inference on validation samples
from collections import defaultdict

correct = 0
total = 0
confusion = defaultdict(lambda: defaultdict(int))

for sample in val_split[:50]:  # Test on 50 samples
    true_class = sample['target_class']
    msgs = sample['messages'][:2]  # System + user only
    
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs, max_new_tokens=128, temperature=0.1,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    try:
        pred = json.loads(response.strip())
        pred_class = pred.get('class', 'UNKNOWN').upper()
    except:
        pred_class = 'UNKNOWN'
    
    confusion[true_class][pred_class] += 1
    if pred_class == true_class:
        correct += 1
    total += 1

accuracy = correct / total * 100
print(f"\nAccuracy: {accuracy:.1f}% ({correct}/{total})")
print(f"\nConfusion Matrix:")
classes = ['CHAINSAW', 'GUNSHOT', 'VEHICLE', 'AMBIENT']
print(f"{'':>12}", *[f'{c:>10}' for c in classes])
for true in classes:
    row = [confusion[true][pred] for pred in classes]
    print(f"{true:>12}", *[f'{v:>10}' for v in row])

## 8. Export for Edge Deployment

Export to GGUF format for Ollama on Raspberry Pi.

In [ ]:
# Save merged model
EXPORT_DIR = "./eco_mesh_model"
model.save_pretrained_merged(
    EXPORT_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to {EXPORT_DIR}")

In [ ]:
# Export to GGUF (INT4 quantized for edge)
model.save_pretrained_gguf(
    "eco_mesh_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Good balance of quality vs size
)
print("GGUF export complete!")

# Show file size
import glob
for f in glob.glob("eco_mesh_gguf/*.gguf"):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}: {size_mb:.0f} MB")

In [ ]:
# Create Ollama Modelfile
modelfile_content = """FROM ./eco_mesh_gguf/unsloth.Q4_K_M.gguf

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_ctx 2048

SYSTEM """You are an Environmental Sound Analyst deployed in a wildlife protection zone. Classify audio segments into: CHAINSAW, GUNSHOT, VEHICLE, or AMBIENT. Respond with JSON only: {"class": "<CATEGORY>", "confidence": <0.0-1.0>, "reasoning": "<brief>"}""" 
"""

with open("eco_mesh_gguf/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created!")
print("\nTo deploy on Pi:")
print("  1. Copy eco_mesh_gguf/ to your Pi")
print("  2. ollama create eco-mesh -f Modelfile")
print("  3. Update config.yaml: model: 'eco-mesh'")

## 9. Download Model

Download the GGUF file to deploy on your edge device.

In [ ]:
# Zip for download
!zip -r eco_mesh_model.zip eco_mesh_gguf/

from google.colab import files
files.download('eco_mesh_model.zip')
print("Download started! Transfer to your Pi and run:")
print("  ollama create eco-mesh -f Modelfile")